In [0]:
# ==========================================
# Notebook: 02_silver_cleaning
#
# Project:
# Pharma Sales Medallion Pipeline
#
# Ticket:
# BTSA-002
#
# Purpose:
# Transform Bronze data into clean,
# standardized Silver tables.
#
# Author:
# Jabir Ahmed
# ==========================================

# Business Context

The Bronze layer preserves the client's raw source data.

This notebook performs data cleaning and standardization to produce
analytics-ready datasets for downstream reporting.

The Silver layer introduces transformations while maintaining
data integrity.

In [0]:
from pyspark.sql import functions as F

In [0]:
prescriptions_df = spark.table("bronze_perscriptions")
products_df = spark.table("bronze_products")
providers_df = spark.table("bronze_providers")
regions_df = spark.table("bronze_regions") 


In [0]:
display(prescriptions_df)

prescriptions_df.printSchema()

print(prescriptions_df.count())

In [0]:
display(products_df)

products_df.printSchema()

print(products_df.count())
display(providers_df)

providers_df.printSchema()

print(providers_df.count())
display(regions_df)

regions_df.printSchema()

print(regions_df.count())

In [0]:
silver_prescriptions = (
    prescriptions_df
        .withColumn("prescription_id", F.trim(F.col("prescription_id")))
        .withColumn("provider_id", F.trim(F.col("provider_id")))
        .withColumn("product_id", F.trim(F.col("product_id")))
        .withColumn("region_id", F.trim(F.col("region_id")))
        .withColumn("prescription_date", F.to_date(F.col("prescription_date")))
        .withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("gross_sales", F.col("gross_sales").cast("double"))
        .withColumn("discount", F.col("discount").cast("double"))
        .withColumn("net_sales", F.round(F.col("gross_sales") - F.col("discount"), 2))
        .dropDuplicates(["prescription_id"])
)
display(silver_prescriptions)
silver_products = (
    products_df
        .withColumn("product_id", F.trim(F.col("product_id")))
        .withColumn("product_name", F.trim(F.col("product_name")))
        .withColumn("therapeutic_area", F.trim(F.col("therapeutic_area")))
        .withColumn("launch_year", F.col("launch_year").cast("int"))
        .withColumn("list_price", F.col("list_price").cast("double"))
        .dropDuplicates(["product_id"])
)

display(silver_products)
silver_providers = (
    providers_df
        .withColumn("provider_id", F.trim(F.col("provider_id")))
        .withColumn("provider_name", F.trim(F.col("provider_name")))
        .withColumn("specialty", F.trim(F.col("specialty")))
        .withColumn("years_practicing", F.col("years_practicing").cast("int"))
        .dropDuplicates(["provider_id"])
)

display(silver_providers)
silver_regions = (
    regions_df
        .withColumn("region_id", F.trim(F.col("region_id")))
        .withColumn("region_name", F.trim(F.col("region_name")))
        .withColumn("sales_director", F.trim(F.col("sales_director")))
        .dropDuplicates(["region_id"])
)

display(silver_regions)

In [0]:
print("Bronze prescriptions:", prescriptions_df.count())
print("Silver prescriptions:", silver_prescriptions.count())

print("Bronze products:", products_df.count())
print("Silver products:", silver_products.count())

print("Bronze providers:", providers_df.count())
print("Silver providers:", silver_providers.count())

print("Bronze regions:", regions_df.count())
print("Silver regions:", silver_regions.count())

In [0]:
silver_prescriptions.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_prescriptions.columns
]).show()

In [0]:
missing_products = (
    silver_prescriptions.alias("p")
    .join(silver_products.alias("pr"), "product_id", "left_anti")
)

missing_providers = (
    silver_prescriptions.alias("p")
    .join(silver_providers.alias("pv"), "provider_id", "left_anti")
)

missing_regions = (
    silver_prescriptions.alias("p")
    .join(silver_regions.alias("r"), "region_id", "left_anti")
)

print("Missing products:", missing_products.count())
print("Missing providers:", missing_providers.count())
print("Missing regions:", missing_regions.count())

In [0]:
revenue_check = (
    silver_prescriptions
        .withColumn("expected_net_sales", F.round(F.col("gross_sales") - F.col("discount"), 2))
        .withColumn("net_sales_diff", F.round(F.col("net_sales") - F.col("expected_net_sales"), 2))
        .filter(F.col("net_sales_diff") != 0)
)

print("Revenue mismatches:", revenue_check.count())

In [0]:
# write silver delta tables

silver_prescriptions.write.format("delta").mode("overwrite").saveAsTable(f"silver_prescriptions")
silver_products.write.format("delta").mode("overwrite").saveAsTable(f"silver_products")
silver_providers.write.format("delta").mode("overwrite").saveAsTable(f"silver_providers")
silver_regions.write.format("delta").mode("overwrite").saveAsTable(f"silver_regions")

In [0]:
spark.sql(f"SELECT COUNT(*) AS silver_prescriptions_count FROM silver_prescriptions").show()
spark.sql(f"SELECT COUNT(*) AS silver_products_count FROM silver_products").show()
spark.sql(f"SELECT COUNT(*) AS silver_providers_count FROM silver_providers").show()
spark.sql(f"SELECT COUNT(*) AS silver_regions_count FROM silver_regions").show()

# Summary

Completed Silver cleaning for prescriptions, products, providers, and regions.

Transformations performed:
- Trimmed ID and text fields
- Cast dates and numeric fields
- Removed duplicate business keys
- Recalculated net sales
- Validated foreign key relationships
- Saved cleaned outputs as Silver Delta tables

Created tables:
- silver_prescriptions
- silver_products
- silver_providers
- silver_regions